In [1]:
import pandas as pd
import datetime as dt
from pathlib import Path
import numpy as np

In [2]:
dirty_dir = Path.home() / "Downloads/churn_analysis_data/churn_analysis_data/dirty"
clean_dir = Path.home() / "Downloads/churn_analysis_data/churn_analysis_data/cleaned"
clean_dir.mkdir(parents=True, exist_ok=True)

customers = pd.read_csv(dirty_dir / "customers.csv")
subscriptions = pd.read_csv(dirty_dir / "subscriptions.csv")
subscription_events = pd.read_csv(dirty_dir / "subscription_events.csv")
invoices = pd.read_csv(dirty_dir / "invoices.csv")
support_tickets = pd.read_csv(dirty_dir / "support_tickets.csv")
usage_events = pd.read_csv(dirty_dir / "usage_events.csv")

tables = {
    "customers": customers, 
    "subscriptions": subscriptions,
    "subscription_events": subscription_events, 
    "invoices": invoices,
    "support_tickets": support_tickets, 
    "usage_events": usage_events
}


In [5]:
for name, df in tables.items():
    print(f"{name}: {df.shape}")
    print(f"{name}: {df.duplicated().sum():,} exact duplicate rows")

customers: (5025, 6)
customers: 25 exact duplicate rows
subscriptions: (5025, 10)
subscriptions: 25 exact duplicate rows
subscription_events: (1389, 8)
subscription_events: 7 exact duplicate rows
invoices: (36624, 11)
invoices: 182 exact duplicate rows
support_tickets: (4920, 10)
support_tickets: 24 exact duplicate rows
usage_events: (469400, 6)
usage_events: 2,335 exact duplicate rows


In [6]:
for name, df in tables.items():
    before = len(tables[name])

    tables[name] = tables[name].drop_duplicates().copy()

    after = len(tables[name])

    print(f"{name}: removed {before - after:,} rows "
            f"({before:,} -> {after:,})")
for name, df in tables.items():
    print(f"{name}: {df.duplicated().sum():,} exact duplicates")    

customers: removed 25 rows (5,025 -> 5,000)
subscriptions: removed 25 rows (5,025 -> 5,000)
subscription_events: removed 7 rows (1,389 -> 1,382)
invoices: removed 182 rows (36,624 -> 36,442)
support_tickets: removed 24 rows (4,920 -> 4,896)
usage_events: removed 2,335 rows (469,400 -> 467,065)
customers: 0 exact duplicates
subscriptions: 0 exact duplicates
subscription_events: 0 exact duplicates
invoices: 0 exact duplicates
support_tickets: 0 exact duplicates
usage_events: 0 exact duplicates


In [7]:
for name, df in tables.items():
    print(f"\n{name}")
    print(
        pd.DataFrame({
            "missing_count": df.isna().sum(),
            "missing_pct": (df.isna().mean() * 100).round(2)
        })
    )


customers
                     missing_count  missing_pct
customer_id                      0          0.0
signup_date                      0          0.0
acquisition_channel              0          0.0
country                          0          0.0
industry                       100          2.0
company_size                   100          2.0

subscriptions
                     missing_count  missing_pct
subscription_id                  0         0.00
customer_id                      0         0.00
start_date                       0         0.00
end_date                      3654        73.08
subscription_status              0         0.00
billing_cycle                    0         0.00
plan                             0         0.00
monthly_price                    0         0.00
churn_type                    3654        73.08
cancellation_reason           3681        73.62

subscription_events
                 missing_count  missing_pct
event_id                     0         0.00
s

In [8]:
tables["customers"][["industry", "company_size"]] = tables["customers"][["industry", "company_size"]].fillna("Unknown")                      

In [9]:
print(tables["customers"][["industry", "company_size"]].isna().sum())

industry        0
company_size    0
dtype: int64


In [11]:
subs = tables["subscriptions"]

mask = (
    (subs["subscription_status"] == "Cancelled") & (subs["cancellation_reason"].isna())
)

print("Missing reasons among cancelled subscriptions:", mask.sum())

subs.loc[mask, "cancellation_reason"] = "Unknown"

Missing reasons among cancelled subscriptions: 27


In [12]:
print(
    subs.groupby("subscription_status")["cancellation_reason"]
    .value_counts(dropna=False)
)

subscription_status  cancellation_reason         
Active               NaN                             3654
Cancelled            Acquisition / Poor Fit           682
                     Repeated Payment Failure         294
                     Other / Changed Requirements     248
                     Poor Support Experience           95
                     Unknown                           27
Name: count, dtype: int64


In [13]:
events = tables["subscription_events"]
subs = tables["subscriptions"]

# Preserve the original plan before cleaning its capitalization
initial_plans = subs.set_index("subscription_id")["plan"].str.title()

# Sort events chronologically within each subscription
events["event_date_parsed"] = pd.to_datetime(
    events["event_date"],
    format="mixed"
)

events = events.sort_values(
    ["subscription_id", "event_date_parsed", "event_id"]
).copy()

# Recover the previous plan from the preceding event
events["recovered_previous_plan"] = (
    events.groupby("subscription_id")["new_plan"].shift(1)
)

# For the first event, use the subscription's original plan
events["recovered_previous_plan"] = (
    events["recovered_previous_plan"].fillna(
        events["subscription_id"].map(initial_plans)
    )
)

# Fill only missing previous_plan values
mask = events["previous_plan"].isna()

events.loc[mask, "previous_plan"] = (
    events.loc[mask, "recovered_previous_plan"]
)

# Remove temporary columns
events = events.drop(
    columns=["event_date_parsed", "recovered_previous_plan"]
)

tables["subscription_events"] = events

In [14]:
print(
    "Missing previous_plan:",
    tables["subscription_events"]["previous_plan"].isna().sum()
)

print(
    "Rows:",
    len(tables["subscription_events"])
)

Missing previous_plan: 0
Rows: 1382


In [15]:
inv = tables["invoices"]

print(
    pd.crosstab(
        inv["retry_count"].fillna(-1),
        inv["payment_status"],
        dropna=False
    )
)

print(
    inv.loc[inv["retry_count"].isna(), [
        "payment_status",
        "failure_reason",
        "recovered",
        "last_attempt_date"
    ]].head(15)
)

payment_status  Failed   Paid  Pending  failed  paid  pending
retry_count                                                  
-1.0                 6    321       27       0    10        0
 0.0                 0  31422     2400       0   621       54
 1.0                 0    661        0       0    23        0
 2.0                 0    372        0       0     8        0
 3.0               293    211        0       3    10        0
     payment_status failure_reason  recovered last_attempt_date
31             Paid            NaN      False        2026-06-10
199            Paid            NaN      False        2026-04-06
312            Paid            NaN      False        2025-12-25
600            Paid            NaN      False        2024-12-29
666            Paid            NaN      False        2025-11-29
748         Pending            NaN      False               NaN
813            Paid            NaN      False        2025-03-19
895            Paid            NaN      False        2

In [16]:
failed_attempt = (
    inv["retry_count"].gt(0)
    | inv["failure_reason"].notna()
)

missing_failure_reason = (
    failed_attempt
    & inv["failure_reason"].isna()
)

print(
    "Missing reasons with evidence of payment failure:",
    missing_failure_reason.sum()
)

Missing reasons with evidence of payment failure: 32


In [17]:
inv = tables["invoices"]

status = inv["payment_status"].str.strip().str.lower()

failure_evidence = (
    status.eq("failed")
    | inv["retry_count"].gt(0)
    | inv["recovered"].eq(True)
)

mask = failure_evidence & inv["failure_reason"].isna()

print("Missing failure reasons to label:", mask.sum())

inv.loc[mask, "failure_reason"] = "Unknown"

Missing failure reasons to label: 32


In [18]:
print("Missing retry counts:", inv["retry_count"].isna().sum())

print(
    "Missing failure reasons with failure evidence:",
    (
        failure_evidence
        & inv["failure_reason"].isna()
    ).sum()
)

print(
    "Unknown failure reasons:",
    inv["failure_reason"].eq("Unknown").sum()
)

Missing retry counts: 364
Missing failure reasons with failure evidence: 0
Unknown failure reasons: 32


In [19]:
tickets = tables["support_tickets"]

# Convert timestamps to datetime
tickets["created_at"] = pd.to_datetime(tickets["created_at"])
tickets["resolved_at"] = pd.to_datetime(tickets["resolved_at"])

# Identify resolved tickets with missing resolution times
mask = (
    tickets["ticket_status"].eq("Resolved")
    & tickets["resolution_time_hours"].isna()
    & tickets["resolved_at"].notna()
)

print("Resolution times to recover:", mask.sum())

# Calculate resolution duration in hours
tickets.loc[mask, "resolution_time_hours"] = (
    (
        tickets.loc[mask, "resolved_at"]
        - tickets.loc[mask, "created_at"]
    ).dt.total_seconds() / 3600
)

Resolution times to recover: 97


In [20]:
print(
    "Missing resolution times:",
    tickets["resolution_time_hours"].isna().sum()
)

print(
    "Resolved tickets with missing resolution times:",
    tickets.loc[
        tickets["ticket_status"] == "Resolved",
        "resolution_time_hours"
    ].isna().sum()
)

print(
    "Negative resolution times:",
    (tickets["resolution_time_hours"] < 0).sum()
)

Missing resolution times: 35
Resolved tickets with missing resolution times: 0
Negative resolution times: 23


In [21]:
usage = tables["usage_events"]

print(
    pd.crosstab(
        usage["event_type"].str.strip().str.lower(),
        usage["feature_category"]
    )
)

feature_category       Analytics  Collaboration  Dashboard  Integrations  \
event_type                                                                 
alert configured           31873              0          0             0   
custom metric created      36354              0          0             0   
dashboard viewed               0              0     174034             0   
data exported                  0              0          0             0   
integration connected          0              0          0         18363   
integration used               0              0          0         55110   
report generated               0              0          0             0   
team member invited            0          22858          0             0   

feature_category       Reporting  
event_type                        
alert configured               0  
custom metric created          0  
dashboard viewed               0  
data exported              36343  
integration connected        

In [22]:
# Normalize event types temporarily for matching
normalized_type = usage["event_type"].str.strip().str.lower()

# Learn the most common category for each event type
category_mapping = (
    usage.loc[usage["feature_category"].notna()]
    .assign(normalized_type=normalized_type)
    .groupby("normalized_type")["feature_category"]
    .agg(lambda x: x.mode().iloc[0])
)

# Fill only missing feature categories
mask = usage["feature_category"].isna()

usage.loc[mask, "feature_category"] = (
    normalized_type.loc[mask].map(category_mapping)
)

In [23]:
print("Missing feature categories:", usage["feature_category"].isna().sum())

print("Missing session durations:", usage["session_duration"].isna().sum())

print(
    "Negative session durations:",
    (usage["session_duration"] < 0).sum()
)

Missing feature categories: 0
Missing session durations: 4671
Negative session durations: 924


In [24]:
categorical_columns = {
    "customers": [
        "acquisition_channel",
        "country",
        "industry",
        "company_size"
    ],
    "subscriptions": [
        "subscription_status",
        "billing_cycle",
        "plan",
        "churn_type",
        "cancellation_reason"
    ],
    "subscription_events": [
        "event_type",
        "previous_plan",
        "new_plan"
    ],
    "invoices": [
        "payment_status",
        "failure_reason"
    ],
    "support_tickets": [
        "issue_category",
        "priority",
        "ticket_status"
    ],
    "usage_events": [
        "event_type",
        "feature_category"
    ]
}

In [25]:
for table_name, columns in categorical_columns.items():
    df = tables[table_name]

    for col in columns:
        df[col] = df[col].str.strip().str.title()

In [26]:
tables["customers"]["country"] = (
    tables["customers"]["country"].replace({
        "Uk": "UK",
        "Us": "US"
    })
)

In [27]:
for table_name, columns in categorical_columns.items():
    print(f"\n--- {table_name} ---")

    for col in columns:
        print(f"\n{col}:")
        print(tables[table_name][col].value_counts(dropna=False))


--- customers ---

acquisition_channel:
acquisition_channel
Paid Search       1394
Organic Search    1103
Referral           823
Partner            612
Social Media       576
Outbound Sales     492
Name: count, dtype: int64

country:
country
US           2205
Canada        849
UK            641
Germany       525
France        441
Australia     339
Name: count, dtype: int64

industry:
industry
Technology               1159
Retail                    946
Professional Services     730
Financial Services        609
Healthcare                525
Manufacturing             424
Education                 310
Other                     197
Unknown                   100
Name: count, dtype: int64

company_size:
company_size
11-50       1445
51-200      1170
1-10        1103
201-500      642
501-1000     334
1001+        206
Unknown      100
Name: count, dtype: int64

--- subscriptions ---

subscription_status:
subscription_status
Active       3654
Cancelled    1346
Name: count, dtype: int64

billin

In [28]:
inv = tables["invoices"]

negative_mask = inv["amount"] < 0

print("Negative invoice amounts:", negative_mask.sum())

inv.loc[negative_mask, "amount"] = (
    inv.loc[negative_mask, "amount"].abs()
)

Negative invoice amounts: 109


In [29]:
tickets = tables["support_tickets"]

negative_mask = tickets["resolution_time_hours"] < 0

print("Negative resolution times:", negative_mask.sum())

tickets.loc[negative_mask, "resolution_time_hours"] = (
    (
        tickets.loc[negative_mask, "resolved_at"]
        - tickets.loc[negative_mask, "created_at"]
    ).dt.total_seconds() / 3600
)

Negative resolution times: 23


In [32]:
usage = tables["usage_events"]

negative_mask = usage["session_duration"] < 0

print("Negative session durations:", negative_mask.sum())

usage.loc[negative_mask, "session_duration"] = np.nan

Negative session durations: 924


In [33]:
print(
    "Negative invoice amounts:",
    (inv["amount"] < 0).sum()
)

print(
    "Negative resolution times:",
    (tickets["resolution_time_hours"] < 0).sum()
)

print(
    "Negative session durations:",
    (usage["session_duration"] < 0).sum()
)

print(
    "Missing session durations:",
    usage["session_duration"].isna().sum()
)

Negative invoice amounts: 0
Negative resolution times: 0
Negative session durations: 0
Missing session durations: 5595


In [34]:
date_columns = {
    "customers": ["signup_date"],
    "subscriptions": ["start_date", "end_date"],
    "subscription_events": ["event_date"],
    "invoices": [
        "invoice_date",
        "due_date",
        "payment_date",
        "last_attempt_date"
    ],
    "support_tickets": ["created_at", "resolved_at"],
    "usage_events": ["event_timestamp"]
}

for table_name, columns in date_columns.items():
    df = tables[table_name]

    for col in columns:
        df[col] = pd.to_datetime(
            df[col],
            format="mixed",
            errors="coerce"
        )

In [35]:
for table_name, columns in date_columns.items():
    print(f"\n{table_name}")

    for col in columns:
        print(f"{col}: {tables[table_name][col].isna().sum():,} missing")


customers
signup_date: 0 missing

subscriptions
start_date: 0 missing
end_date: 3,654 missing

subscription_events
event_date: 0 missing

invoices
invoice_date: 0 missing
due_date: 0 missing
payment_date: 2,783 missing
last_attempt_date: 2,481 missing

support_tickets
created_at: 0 missing
resolved_at: 35 missing

usage_events
event_timestamp: 0 missing


In [36]:
primary_keys = {
    "customers": "customer_id",
    "subscriptions": "subscription_id",
    "subscription_events": "event_id",
    "invoices": "invoice_id",
    "support_tickets": "ticket_id",
    "usage_events": "event_id"
}

for table_name, pk in primary_keys.items():
    df = tables[table_name]

    print(
        f"{table_name}: "
        f"{df[pk].duplicated().sum():,} duplicate IDs, "
        f"{df[pk].isna().sum():,} missing IDs"
    )

customers: 0 duplicate IDs, 0 missing IDs
subscriptions: 0 duplicate IDs, 0 missing IDs
subscription_events: 0 duplicate IDs, 0 missing IDs
invoices: 0 duplicate IDs, 0 missing IDs
support_tickets: 0 duplicate IDs, 0 missing IDs
usage_events: 0 duplicate IDs, 0 missing IDs


In [38]:
foreign_keys = [
    ("subscriptions", "customer_id", "customers", "customer_id"),
    ("subscription_events", "subscription_id", "subscriptions", "subscription_id"),
    ("invoices", "subscription_id", "subscriptions", "subscription_id"),
    ("support_tickets", "customer_id", "customers", "customer_id"),
    ("usage_events", "customer_id", "customers", "customer_id")
]

for child_table, fk, parent_table, pk in foreign_keys:
    child = tables[child_table]
    parent = tables[parent_table]

    unmatched_count = (
        ~child[fk].isin(parent[pk])
    ).sum()

    print(
        f"{child_table}.{fk}: "
        f"{unmatched_count:,} unmatched records"
    )

subscriptions.customer_id: 0 unmatched records
subscription_events.subscription_id: 0 unmatched records
invoices.subscription_id: 0 unmatched records
support_tickets.customer_id: 0 unmatched records
usage_events.customer_id: 0 unmatched records


In [39]:
subs = tables["subscriptions"]

active_with_end_date = (
    subs["subscription_status"].eq("Active")
    & subs["end_date"].notna()
).sum()

cancelled_without_end_date = (
    subs["subscription_status"].eq("Cancelled")
    & subs["end_date"].isna()
).sum()

end_before_start = (
    subs["end_date"] < subs["start_date"]
).sum()

print("Active subscriptions with end dates:", active_with_end_date)
print("Cancelled subscriptions without end dates:", cancelled_without_end_date)
print("Subscription end dates before start dates:", end_before_start)

Active subscriptions with end dates: 0
Cancelled subscriptions without end dates: 0
Subscription end dates before start dates: 0


In [40]:
subs = tables["subscriptions"]

subscription_dates = subs[
    ["subscription_id", "customer_id", "start_date", "end_date"]
].copy()

subscription_dates["effective_end"] = (
    subscription_dates["end_date"]
    .fillna(pd.Timestamp("2026-08-01"))
)


def check_event_dates(df, parent, date_col, join_col):
    merged = df.merge(
        parent,
        on=join_col,
        how="left",
        validate="many_to_one"
    )

    before_start = (
        merged[date_col] < merged["start_date"]
    ).sum()

    after_end = (
        merged[date_col] > merged["effective_end"]
    ).sum()

    return before_start, after_end


# Subscription events
print(
    "Subscription events:",
    check_event_dates(
        tables["subscription_events"],
        subscription_dates[
            ["subscription_id", "start_date", "effective_end"]
        ],
        "event_date",
        "subscription_id"
    )
)

# Invoices
print(
    "Invoices:",
    check_event_dates(
        tables["invoices"],
        subscription_dates[
            ["subscription_id", "start_date", "effective_end"]
        ],
        "invoice_date",
        "subscription_id"
    )
)

# Usage events
print(
    "Usage events:",
    check_event_dates(
        tables["usage_events"],
        subscription_dates[
            ["customer_id", "start_date", "effective_end"]
        ],
        "event_timestamp",
        "customer_id"
    )
)

# Support tickets
print(
    "Support tickets:",
    check_event_dates(
        tables["support_tickets"],
        subscription_dates[
            ["customer_id", "start_date", "effective_end"]
        ],
        "created_at",
        "customer_id"
    )
)

Subscription events: (0, 0)
Invoices: (0, 0)
Usage events: (0, 0)
Support tickets: (0, 0)


In [41]:
for name, df in tables.items():
    output_path = clean_dir / f"{name}.csv"
    df.to_csv(output_path, index=False)
    print(f"Saved {name}: {len(df):,} rows -> {output_path}")

Saved customers: 5,000 rows -> /Users/bryantcortez/Downloads/churn_analysis_data/churn_analysis_data/cleaned/customers.csv
Saved subscriptions: 5,000 rows -> /Users/bryantcortez/Downloads/churn_analysis_data/churn_analysis_data/cleaned/subscriptions.csv
Saved subscription_events: 1,382 rows -> /Users/bryantcortez/Downloads/churn_analysis_data/churn_analysis_data/cleaned/subscription_events.csv
Saved invoices: 36,442 rows -> /Users/bryantcortez/Downloads/churn_analysis_data/churn_analysis_data/cleaned/invoices.csv
Saved support_tickets: 4,896 rows -> /Users/bryantcortez/Downloads/churn_analysis_data/churn_analysis_data/cleaned/support_tickets.csv
Saved usage_events: 467,065 rows -> /Users/bryantcortez/Downloads/churn_analysis_data/churn_analysis_data/cleaned/usage_events.csv


In [3]:
invoices = pd.read_csv(clean_dir / "invoices.csv")
support_tickets = pd.read_csv(clean_dir / "support_tickets.csv")

invoices["recovered"] = invoices["recovered"].astype(int)
support_tickets["sla_breached"] = support_tickets["sla_breached"].astype(int)

In [7]:
invoices["recovered"].value_counts()

recovered
0    35145
1     1297
Name: count, dtype: int64

In [8]:
support_tickets["sla_breached"].value_counts()

sla_breached
1    2561
0    2335
Name: count, dtype: int64

In [9]:
invoices.to_csv(clean_dir / "invoices.csv", index=False)
support_tickets.to_csv(clean_dir / "support_tickets.csv", index=False)

In [12]:
invoices = pd.read_csv(clean_dir / "invoices.csv")
subscriptions = pd.read_csv(clean_dir / "subscriptions.csv")

missing_ids = ~invoices["subscription_id"].isin(
    subscriptions["subscription_id"]
)

print("Orphan invoices:", missing_ids.sum())

Orphan invoices: 0


In [13]:
subscriptions = pd.read_csv(clean_dir / "subscriptions.csv")

active_subscriptions = subscriptions[
    subscriptions["subscription_status"] == "Active"
].copy()

print(active_subscriptions.shape)

(3654, 10)


In [14]:
active_subscriptions.to_csv(
    clean_dir / "active_subscriptions.csv",
    index=False
)

In [15]:
subscriptions = pd.read_csv(clean_dir / "subscriptions.csv")

active_subscriptions = subscriptions[
    subscriptions["subscription_status"] == "Active"
].copy()

active_subscriptions.to_csv(
    clean_dir / "active_subscriptions.csv",
    index=False,
    na_rep=r"\N"
)

In [16]:
active_subscriptions.to_csv(
    clean_dir / "active_subscriptions.csv",
    index=False
)

In [18]:
df = pd.read_csv(
    "/Users/bryantcortez/Downloads/churn_analysis_data/churn_analysis_data/cleaned/support_tickets.csv"
)

warning_rows = [
    522, 847, 1029, 1380, 1920, 2115, 2531,
    2964, 3100, 3479, 3714, 4321, 4429, 4584
]

print(
    df.iloc[[row - 1 for row in warning_rows]][
        ["ticket_id", "resolution_time_hours"]
    ].to_string(index=False)
)

                           ticket_id  resolution_time_hours
82dc56cc-94ed-4f5a-9a8e-e00b7eb901dc             167.689336
1c1437ef-1f03-48c0-b550-f2088b854008              78.752464
980976ce-6978-492b-aeb8-b691c25f73ad             165.038612
78cddc27-7c76-4ea3-9d1a-4f7a0c1fe902             108.314594
f4da21f1-c54b-46a3-becd-b1759c843e40             162.155603
7f7af43e-695f-4d0c-8653-4f76267921e3             101.175328
c7b804a9-4f80-473c-b444-43e098142b04              87.604949
ba1a0694-537f-44b3-8c05-bb2f3defceb6             125.685564
58341a94-95a0-45af-988b-747931073ea1             142.601371
bc7f509d-9b0d-42b5-9d03-eed829417c45             117.057792
afcc3b6e-8bd9-4632-b77d-edd6c9a2c974              80.802978
504b2fb2-94ec-421d-9505-1cf95c2c9fc1             115.277300
ab26e1ca-82d8-4576-8eba-8895f448748e             116.983950
a99b5d48-2664-4410-93bf-0a53ba66c2b7             120.031449
